In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Configuración de consistencia
np.random.seed(42)
num_productos = 50
dias_totales = 365
fecha_inicio = datetime(2025, 1, 1)

# 1. GENERAR DIM_PRODUCTOS
productos = []
for i in range(1, num_productos + 1):
    productos.append({
        'COD_ARTICULO': int(1000 + i), # ID Numérico entero
        'NOM_ARTICULO': f'Producto Alfa {i}',
        'CATEGORIA': np.random.choice(['Electrónica', 'Repuestos', 'Hogar', 'Herramientas']),
        'COSTO_UNITARIO': round(np.random.uniform(5.0, 150.0), 2)
    })
df_productos = pd.DataFrame(productos)
df_productos.to_csv('Dim_Productos.csv', index=False)

# 2. GENERAR MOVIMIENTOS DE INVENTARIO (Fact_Movimientos)
movimientos = []
fechas = [fecha_inicio + timedelta(days=x) for x in range(dias_totales)]

for fecha in fechas:
    # Formato numérico AAAAMMDD exigido por la arquitectura transaccional
    fecha_num = int(fecha.strftime('%Y%m%d')) 
    
    # Seleccionar una muestra aleatoria de productos que se movieron ese día
    productos_del_dia = np.random.choice(df_productos['COD_ARTICULO'], size=np.random.randint(10, 30), replace=False)
    
    for prod in productos_del_dia:
        tipo_mov = np.random.choice(['E', 'S'], p=[0.3, 0.7]) # E=Entrada (Compra), S=Salida (Venta)
        cant = np.random.randint(1, 50)
        
        movimientos.append({
            'FECHA_MOV': float(fecha_num), # Forzamos DECIMAL(8,0) simulado en tipo flotante/numérico puro
            'COD_ARTICULO': int(prod),
            'COD_DEPOSITO': np.random.choice([1, 2]),
            'TIPO_MOV': tipo_mov,
            'CANT_MOV': float(cant) # Equivalente a DECIMAL(11,2)
        })

df_movimientos = pd.DataFrame(movimientos)
df_movimientos.to_csv('Fact_Movimientos_Inventario.csv', index=False)

# 3. GENERAR VENTAS HISTÓRICAS (Para el cálculo DAX del Stock de Seguridad)
ventas = []
for fecha in fechas:
    fecha_num = int(fecha.strftime('%Y%m%d'))
    productos_vendidos = np.random.choice(df_productos['COD_ARTICULO'], size=np.random.randint(15, 35), replace=False)
    
    for prod in productos_vendidos:
        cant_vendida = np.random.randint(1, 15)
        ventas.append({
            'FECHA_VENTA': int(fecha_num),
            'COD_ARTICULO': int(prod),
            'CANT_VENDIDA': int(cant_vendida)
        })
df_ventas = pd.DataFrame(ventas)
df_ventas.to_csv('Fact_Ventas.csv', index=False)

print("¡Archivos CSV creados con éxito! 'Dim_Productos.csv', 'Fact_Movimientos_Inventario.csv' y 'Fact_Ventas.csv' listos.")

¡Archivos CSV creados con éxito! 'Dim_Productos.csv', 'Fact_Movimientos_Inventario.csv' y 'Fact_Ventas.csv' listos.


In [3]:
# Leer el CSV crudo que generamos antes
df_mov_crudo = pd.read_csv('Fact_Movimientos_Inventario.csv')

# Crear la columna NETO_MOVIMIENTOS aplicando el CASE WHEN (Multiplicar por -1 si es Salida 'S')
df_mov_crudo['NETO_MOVIMIENTOS'] = df_mov_crudo.apply(
    lambda row: row['CANT_MOV'] if row['TIPO_MOV'] == 'E' else row['CANT_MOV'] * -1, 
    axis=1
)

# Hacer el GROUP BY equivalente al script SQL
df_fact_inventario_final = df_mov_crudo.groupby(['FECHA_MOV', 'COD_ARTICULO', 'COD_DEPOSITO'], as_index=False)['NETO_MOVIMIENTOS'].sum()

# Renombrar columnas para que queden limpias para Power BI
df_fact_inventario_final.rename(columns={'FECHA_MOV': 'FechaKey'}, inplace=True)

# Guardar el archivo final que verdaderamente va a importar Power BI
df_fact_inventario_final.to_csv('Fact_Movimientos_Procesado.csv', index=False)
print("¡Archivo procesado listo para Power BI!")

¡Archivo procesado listo para Power BI!
